# STEW mental workload — full pipeline

Run this top to bottom once your data and config are in place. The code lives in `src/stew_mwl/`; see the repo `README.md` for dataset layout and file names.

**What runs here:** build a subject manifest → leave-one-subject-out training for the proposed model (VAE + CBAM + BiLSTM) → same splits for PSD–SVM, CNN, and BLSTM–LSTM baselines → ablations → optional sensitivity sweeps (if enabled in YAML) → figures, Grad-CAM, and CSV exports under `outputs/`.

**Classes:** BL (baseline `lo` recording), LW / MW / HW from task ratings 1–3 / 4–6 / 7–9.

**Config:** In the next section, point `CONFIG_FILE` to `configs/default.yaml` for a short run or `full_reproduction.yaml` for the full study (much slower).

### Paths

`ROOT` is the repository root (parent of this `notebooks/` folder) so data and outputs stay out of the notebook directory.

In [ ]:
import sys
from pathlib import Path

# Repo root (not notebooks/)
ROOT = Path("..").resolve()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print("ROOT:", ROOT)

### Imports and versions

In [ ]:
import sys

import numpy as np
import pandas as pd
import tensorflow as tf
from IPython.display import display
from sklearn.metrics import confusion_matrix

from stew_mwl import plotting
from stew_mwl.config import CLASS_NAMES
from stew_mwl.data import build_subject_manifest, set_global_determinism, validate_stew_dataset
from stew_mwl.eval import aggregate_fold_metrics, paired_ttest_vs_full
from stew_mwl.export import (
    export_ablation_summary,
    export_baseline_comparison_summary,
    export_classification_report_and_cm,
    export_dataset_manifest,
    export_experiment_registry,
    export_fold_metrics_and_predictions,
    export_gradcam_summaries,
    export_segmentation_summary,
    export_statistical_tests,
)
from stew_mwl.gradcam import run_gradcam_export_for_checkpoint
from stew_mwl.reports import build_manuscript_tables, write_run_manifest
from stew_mwl.train import (
    run_ablation_variants,
    run_baseline_models,
    run_loso_training,
    run_sensitivity_grids,
)
from stew_mwl.yaml_loader import load_config_from_yaml

print("Python", sys.version.split()[0], "| TensorFlow", tf.__version__)

### Configuration

YAML lives under `configs/`. Always pass `project_root=ROOT` so paths resolve to the repo root, not this folder.

In [ ]:
CONFIG_FILE = ROOT / "configs" / "default.yaml"
# CONFIG_FILE = ROOT / "configs" / "full_reproduction.yaml"

cfg = load_config_from_yaml(CONFIG_FILE, project_root=ROOT)
set_global_determinism(cfg.seed)
cfg.ensure_dirs()

print("config:", CONFIG_FILE.name)
print("classes:", CLASS_NAMES)
print("data_root:", cfg.data_root)
print("outputs:", cfg.output_root)

## Dataset

STEW signals under `data/STEW/` (see README): paired `sub##_lo.txt` / `sub##_hi.txt` plus a ratings file (filename contains `rating`).

In [ ]:
manifest = build_subject_manifest(cfg)
issues = validate_stew_dataset(cfg, manifest)
if issues:
    print("Validation notes:")
    for msg in issues:
        print(" ", msg)
else:
    print("Validation OK.")

display(manifest.head())
manifest.shape, manifest["task_label"].value_counts()

### Manifest and segmentation CSVs

In [ ]:
export_dataset_manifest(manifest, cfg)
export_segmentation_summary(manifest, cfg)
print("dataset_manifest.csv, segmentation_summary.csv →", cfg.csv_dir)

## Proposed model (LOSO)

Trains one fold per held-out subject. Saves metrics, predictions, VAE loss CSV, and `fold_*_proposed.keras` under `outputs/`.

In [ ]:
full_df, loso_splits, predictions = run_loso_training(cfg, manifest, model_name="proposed")
_, full_summary = aggregate_fold_metrics(full_df.to_dict("records"))
full_summary

In [ ]:
export_fold_metrics_and_predictions(full_df, predictions, cfg)
export_classification_report_and_cm(predictions, cfg)
print("Fold metrics, predictions, report, confusion matrix →", cfg.csv_dir)

## Baselines

Same LOSO splits as above: PSD–SVM (hand-crafted features), CNN, BLSTM–LSTM.

In [ ]:
baseline_results = run_baseline_models(cfg, manifest, loso_splits=loso_splits)
export_baseline_comparison_summary(baseline_results, cfg)
for name, df in baseline_results.items():
    print(name, "folds:", len(df))
    if len(df):
        print(df[["subject", "accuracy", "macro_f1"]].head())

## Ablations

Variants turn off VAE, CBAM, bidirectionality, or recurrence (see `train.run_ablation_variants`). The full model’s fold metrics are included for comparison.

In [ ]:
ablation_results = run_ablation_variants(
    cfg, manifest, loso_splits=loso_splits, full_fold_metrics_df=full_df
)
export_ablation_summary(ablation_results, full_df, cfg)
export_statistical_tests(full_df, baseline_results, ablation_results, cfg)
export_experiment_registry(cfg, notes="notebook run")
list(ablation_results.keys())

### Quick check: paired *t* on accuracy

Same pairing as in `statistical_tests.csv` — here just printing *p* for proposed vs BLSTM–LSTM if that baseline ran.

In [ ]:
if "blstm_lstm" in baseline_results and len(baseline_results["blstm_lstm"]):
    p = paired_ttest_vs_full(full_df, baseline_results["blstm_lstm"], "accuracy")
    print("p (proposed vs blstm_lstm, accuracy):", p)

## Sensitivity (optional)

Sweeps latent size, CBAM reduction/kernel, window length, and sequence length. Turn on with `run_sensitivity: true` in YAML. On all 48 subjects this is slow; with `quick_mode: true` only a few subjects are used inside `run_sensitivity_grids`.

In [ ]:
if cfg.run_sensitivity:
    sens = run_sensitivity_grids(cfg, manifest)
    print("Done. DataFrames:", list(sens.keys()))
else:
    print("Skipped (run_sensitivity is false in YAML).")

## Figures

Confusion matrix, VAE losses, baseline and ablation bar charts when the corresponding CSVs exist.

In [ ]:
pred_df = pd.DataFrame(predictions)
if len(pred_df):
    cm = confusion_matrix(
        pred_df["y_true"].values,
        pred_df["y_pred"].values,
        labels=list(range(len(CLASS_NAMES))),
    )
    plotting.plot_confusion_matrix(cm, cfg)
plotting.plot_vae_loss_curves(cfg.csv_dir / "vae_fold_losses.csv", cfg)
plotting.plot_baseline_bar(cfg.csv_dir / "baseline_comparison_summary.csv", cfg)
plotting.plot_ablation_bar(cfg.csv_dir / "ablation_summary.csv", cfg)
print("Figures →", cfg.figures_dir)

## Grad-CAM

Uses the first saved proposed checkpoint. Heatmaps are split into upper vs lower half of the grid as a simple frontal/parietal proxy. Writes `gradcam_*.csv` and optional PNGs.

In [ ]:
ckpts = sorted(cfg.models_dir.glob("fold_*_proposed.keras"))
if ckpts:
    reg_df, samp_df = run_gradcam_export_for_checkpoint(
        cfg, manifest, ckpts[0], max_samples=16, save_example_figure=True
    )
    export_gradcam_summaries(reg_df, samp_df, cfg)
    plotting.plot_gradcam_outputs_on_disk(cfg)
    print("Grad-CAM from", ckpts[0].name)
else:
    print("No checkpoints yet — run the LOSO section first.")

### CSV listing

Everything below `outputs/csv/` after a run:

In [ ]:
for p in sorted(cfg.csv_dir.glob("*.csv")):
    print(p.name)

### Summary tables and manifest

Rolls existing CSVs into `outputs/reports/` and writes `output_manifest.csv` under `outputs/logs/`.

In [ ]:
table_paths = build_manuscript_tables(cfg)
write_run_manifest(cfg, extra={"config_path": str(cfg.config_path or "")})
print(cfg.reports_dir)
for k, v in sorted(table_paths.items()):
    print(" ", k, "→", v.name)

### Done

Tweak YAML, re-run from the section you care about, or clear `outputs/` for a clean slate. For column-level documentation of each CSV, use the README table.